# E7 — Transfer learning modif: backbone DIBEKUKAN + head baru + Adam + CE
Hipotesis: fitur ImageNet sudah sangat umum; untuk CIFAR-10 32x32 mungkin cukup dipakai apa adanya. Modifikasi vs E6:
- Semua `features` EfficientNet-B0 di-`requires_grad=False` (feature extractor beku)
- Head baru: Dropout(0.3) → Linear 1280→128 → ReLU → Linear 128→10
- Output logits + CrossEntropy (bukan sigmoid+BCE)
- Adam lr=0.001 (bukan SGD lr=0.0025)

Konsekuensi: parameter trainable turun drastis (dari ~4 juta jadi ~165 ribu), backprop hanya lewat head → jauh lebih cepat per epoch. Pertanyaannya: berapa akurasi yang hilang (atau justru naik?) dibanding full fine-tune E6.

## Run

In [1]:
from exp_common import *
model = EffNetFrozen()
res = run_experiment(
    'E7_transfer_frozen', model,
    train_files=['data_batch_1', 'data_batch_3', 'data_batch_4', 'data_batch_5'],
    val_files=['data_batch_2'],
    epochs=3, batch_size=64, lr=0.001, optimizer_type='adam',
    loss_mode='ce', augment=True, fix_channels=True)


===== E7_transfer_frozen =====
device=cuda, epochs=3, batch=64, lr=0.001, opt=adam, loss=ce, augment=True, fix_channels=True
params total=4,172,806 trainable=165,258


train=40000 val=10000


epoch 0: train_loss=1.8962 train_acc=0.3256 val_loss=1.7885 val_acc=0.3633 (29.4s, 1360 samples/s)


epoch 1: train_loss=1.8090 train_acc=0.3569 val_loss=1.7627 val_acc=0.3711 (29.9s, 1336 samples/s)


epoch 2: train_loss=1.8013 train_acc=0.3590 val_loss=1.7474 val_acc=0.3831 (31.3s, 1278 samples/s)
peak GPU mem: 94 MB


TEST ACC = 0.3890  (eval 1.4s, train 90.7s)
saved: C:\Users\dgash\Downloads\ml_task_cnn\experiments\results\E7_transfer_frozen.json
